# Decision Classifier

Binary sentence-level classifier: **decision** vs **not decision**.

**Enhancements over the Importance Classifier:**
- Decision-specific handcrafted features — consensus phrases, resolution keywords, hedging density, passive constructions
- **BM25** relevance score against a decision-keyword query (beats raw TF-IDF term weighting)
- **Sentence-BERT (SBERT)** dense embeddings — captures meaning beyond keyword overlap
- Extended model zoo: **XGBoost**, **LightGBM**, **SVM (RBF kernel)**, and a **Stacking Ensemble**
- Extra metrics: **PR-AUC** and **MCC** — better measures for imbalanced binary classification
- **Threshold tuning** — finds the optimal decision boundary instead of defaulting to 0.5

**Pipeline:**
1. Load `data/processed/decision_data.json` (produced by `01_data_preparation.ipynb`)
2. Feature engineering — TF-IDF + BM25 + 17 handcrafted features + SBERT embeddings (optional)
3. Train & evaluate seven classifiers + one stacking ensemble
4. Threshold tuning on the best model
5. Compare with metrics table, confusion matrices, ROC & PR curves
6. Save the best model to `models/decision_classifier.pkl`

## 1. Setup

In [ ]:
import json
import re
import pickle
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from pathlib import Path
from collections import Counter
from matplotlib.patches import Patch

from scipy.sparse import hstack, csr_matrix

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, StackingClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, roc_curve, confusion_matrix, classification_report,
    average_precision_score, precision_recall_curve, matthews_corrcoef
)

# ── Optional packages ──────────────────────────────────────────────────────────
try:
    from xgboost import XGBClassifier
    HAS_XGB = True
except ImportError:
    HAS_XGB = False
    print('XGBoost not installed.  Run:  pip install xgboost')

try:
    from lightgbm import LGBMClassifier
    HAS_LGBM = True
except ImportError:
    HAS_LGBM = False
    print('LightGBM not installed. Run:  pip install lightgbm')

try:
    from rank_bm25 import BM25Okapi
    HAS_BM25 = True
except ImportError:
    HAS_BM25 = False
    print('rank_bm25 not installed.  Run:  pip install rank-bm25')

try:
    from sentence_transformers import SentenceTransformer
    HAS_SBERT = True
except ImportError:
    HAS_SBERT = False
    print('sentence-transformers not installed.  Run:  pip install sentence-transformers')

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid')
PALETTE = ['#4C72B0', '#DD8452', '#55A868', '#C44E52',
           '#8172B2', '#937860', '#DA8BC3', '#8C8C8C']

ROOT      = Path('../')
DATA_PATH = ROOT / 'data' / 'processed' / 'decision_data.json'
MODEL_DIR = ROOT / 'models'
MODEL_DIR.mkdir(parents=True, exist_ok=True)

SEED = 42
np.random.seed(SEED)

print('All imports OK')
print(f'Data path : {DATA_PATH}')
print(f'Model dir : {MODEL_DIR}')
print(f'XGBoost   : {"available" if HAS_XGB  else "missing"}')
print(f'LightGBM  : {"available" if HAS_LGBM else "missing"}')
print(f'BM25      : {"available" if HAS_BM25  else "missing"}')
print(f'SBERT     : {"available" if HAS_SBERT else "missing"}')

## 2. Load & Explore Data

In [ ]:
with open(DATA_PATH, encoding='utf-8') as f:
    raw = json.load(f)

df = pd.DataFrame(raw)
print(f'Total examples : {len(df)}')
print(f'Columns        : {list(df.columns)}')
print(f'Class balance  :')
print(df['label'].value_counts().rename({1: 'Decision (1)', 0: 'Not Decision (0)'}))
df.head(6)

In [ ]:
# ── EDA: class distribution + sentence length ──────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

counts = df['label'].value_counts().sort_index()
imbalance_ratio = counts[0] / counts[1]
bars = axes[0].bar(['Not Decision (0)', 'Decision (1)'], counts.values,
                   color=PALETTE[:2], edgecolor='white', linewidth=1.5, width=0.5)
for bar, v in zip(bars, counts.values):
    axes[0].text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 3,
                 str(v), ha='center', va='bottom', fontweight='bold', fontsize=12)
axes[0].set_title(f'Class Distribution  (imbalance ratio ≈ {imbalance_ratio:.1f}:1)',
                  fontsize=13, fontweight='bold')
axes[0].set_ylabel('Count')
axes[0].set_ylim(0, counts.max() * 1.2)

df['word_count'] = df['text'].str.split().str.len()
for label, color, name in [(0, PALETTE[0], 'Not Decision'), (1, PALETTE[1], 'Decision')]:
    axes[1].hist(df[df['label'] == label]['word_count'], bins=30,
                 alpha=0.65, color=color, label=name, edgecolor='white')
axes[1].set_title('Sentence Length Distribution', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Word Count')
axes[1].set_ylabel('Frequency')
axes[1].legend()

plt.tight_layout()
plt.savefig(MODEL_DIR / 'decision_eda.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Class imbalance ratio (neg:pos) = {imbalance_ratio:.2f}:1')

## 3. Feature Engineering

Four feature groups are built and fused:
- **TF-IDF** (unigrams + bigrams, top 8 000 features) — lexical pattern capture
- **BM25 score** — relevance against a hand-crafted decision keyword query (better than raw TF term weight)
- **Handcrafted linguistic features** (17 features) — decision-specific signals: consensus phrases, resolution keywords, hedging density, passive constructions
- **SBERT embeddings** (384-dim) — dense semantic vectors; captures that *"we went with X"* and *"X was selected"* mean the same thing

In [ ]:
# ── Regex patterns — decision-specific ────────────────────────────────────────
MODAL_VERBS      = re.compile(r'\b(will|would|shall|should|must|need to|going to|have to)\b', re.I)
DECISION_KWS     = re.compile(r'\b(decided|agreed|confirmed|resolved|concluded|chosen|approved|rejected|accepted|finalised|finalized|going with|stick with|settle on|go with|go ahead|move forward)\b', re.I)
CONSENSUS_KWS    = re.compile(r'\b(so we|so the|then we|let us|we will|we are going to|we should)\b', re.I)
RESOLUTION_KWS   = re.compile(r'\b(final|finalize|finalized|conclusion|conclude|concluded|resolved|resolution|confirm|confirmed|settle|settled)\b', re.I)
PASSIVE_DECISION = re.compile(r'\b(was decided|has been decided|was agreed|has been agreed|was chosen|was selected|was approved)\b', re.I)
HEDGING_KWS      = re.compile(r'\b(maybe|perhaps|possibly|might|not sure|probably|seems|appears)\b', re.I)
AGREEMENT_SIGS   = re.compile(r'\b(yeah|yes|right|okay|agreed|exactly|correct|sure|absolutely|definitely)\b', re.I)
NEGATION_KWS     = re.compile(r'\b(not|no|never|without|dont|doesnt|didnt|wont|cant|cannot)\b', re.I)
FIRST_PERSON     = re.compile(r'\b(I|we|our|my|us)\b')
QUESTION_RE      = re.compile(r'\?')


def extract_handcrafted(texts: list) -> np.ndarray:
    """Return (n_samples, 17) float array of decision-focused linguistic features."""
    features = []
    for text in texts:
        words = text.split()
        n = max(len(words), 1)
        row = [
            len(MODAL_VERBS.findall(text)),            # modal verb count
            len(DECISION_KWS.findall(text)),           # decision keyword count
            len(CONSENSUS_KWS.findall(text)),          # consensus phrase count
            len(RESOLUTION_KWS.findall(text)),         # resolution keyword count
            int(bool(PASSIVE_DECISION.search(text))),  # passive decision construction
            len(HEDGING_KWS.findall(text)),            # hedging word count
            len(AGREEMENT_SIGS.findall(text)),         # agreement signal count
            len(NEGATION_KWS.findall(text)),           # negation count
            len(FIRST_PERSON.findall(text)),           # first-person pronoun count
            int(bool(QUESTION_RE.search(text))),       # is a question?
            len(words),                                # sentence length (words)
            len(text),                                 # sentence length (chars)
            len(MODAL_VERBS.findall(text)) / n,        # modal density
            len(HEDGING_KWS.findall(text)) / n,        # hedging density (low in decisions)
            sum(1 for w in words if w[0].isupper() and len(w) > 1) / n,  # capital ratio
            int(text.endswith('.')),                   # ends with period
            text.count(',') / n,                       # comma density
        ]
        features.append(row)
    return np.array(features, dtype=float)


FEATURE_NAMES_HANDCRAFTED = [
    'modal_count', 'decision_kw_count', 'consensus_kw_count', 'resolution_kw_count',
    'is_passive_decision', 'hedging_count', 'agreement_signal_count', 'negation_count',
    'first_person_count', 'is_question', 'word_count', 'char_count',
    'modal_density', 'hedging_density', 'capital_ratio', 'ends_with_period', 'comma_density',
]

print(f'Handcrafted features ({len(FEATURE_NAMES_HANDCRAFTED)}): {FEATURE_NAMES_HANDCRAFTED}')

In [ ]:
texts  = df['text'].tolist()
labels = df['label'].values

# ── TF-IDF ─────────────────────────────────────────────────────────────────────
tfidf = TfidfVectorizer(
    ngram_range=(1, 2),
    max_features=8_000,
    sublinear_tf=True,
    min_df=2,
    strip_accents='unicode',
)
X_tfidf = tfidf.fit_transform(texts)

# ── BM25 ───────────────────────────────────────────────────────────────────────
# Score each sentence against a decision-keyword query.
# BM25 penalises very short sentences and very long ones, unlike raw TF.
DECISION_QUERY = [
    'decided', 'agreed', 'confirmed', 'go', 'with', 'conclude',
    'final', 'resolved', 'approved', 'selected', 'chosen',
]
if HAS_BM25:
    tokenized_corpus = [t.lower().split() for t in texts]
    bm25_index = BM25Okapi(tokenized_corpus)
    bm25_scores = bm25_index.get_scores(DECISION_QUERY)
    bm25_scores = (bm25_scores - bm25_scores.min()) / (bm25_scores.max() - bm25_scores.min() + 1e-9)
    print(f'BM25 scores  : min={bm25_scores.min():.4f}  max={bm25_scores.max():.4f}  mean={bm25_scores.mean():.4f}')
else:
    bm25_scores = np.zeros(len(texts))
    print('BM25 not available — using zero placeholder.')

# ── Handcrafted ────────────────────────────────────────────────────────────────
X_hand        = extract_handcrafted(texts)
X_hand_scaled = StandardScaler().fit_transform(X_hand)

print(f'TF-IDF shape     : {X_tfidf.shape}')
print(f'BM25 shape       : {bm25_scores.shape}')
print(f'Handcrafted shape: {X_hand.shape}')

In [ ]:
# ── SBERT Sentence Embeddings ──────────────────────────────────────────────────
# all-MiniLM-L6-v2: 384-dim vectors, ~80 MB download on first run.
# Captures semantic similarity — "we settled on X" clusters near "X was chosen".
if HAS_SBERT:
    print('Loading SBERT model (all-MiniLM-L6-v2)...')
    sbert_model = SentenceTransformer('all-MiniLM-L6-v2')
    X_sbert = sbert_model.encode(texts, batch_size=64, show_progress_bar=True,
                                  convert_to_numpy=True)
    SBERT_DIM = X_sbert.shape[1]
    print(f'SBERT shape: {X_sbert.shape}')
else:
    SBERT_DIM = 0
    X_sbert   = np.zeros((len(texts), 0))
    print('SBERT not available — feature set will be TF-IDF + BM25 + Handcrafted only.')

In [ ]:
# ── Feature Fusion ─────────────────────────────────────────────────────────────
# Stack all feature groups into one sparse matrix.
# csr_matrix wraps dense arrays so scipy hstack can handle them.
bm25_col = csr_matrix(bm25_scores.reshape(-1, 1))
hand_col  = csr_matrix(X_hand_scaled)

if HAS_SBERT and SBERT_DIM > 0:
    X_fused = hstack([X_tfidf, bm25_col, hand_col, csr_matrix(X_sbert)])
    feature_description = f'TF-IDF({X_tfidf.shape[1]}) + BM25(1) + Handcrafted({X_hand.shape[1]}) + SBERT({SBERT_DIM})'
else:
    X_fused = hstack([X_tfidf, bm25_col, hand_col])
    feature_description = f'TF-IDF({X_tfidf.shape[1]}) + BM25(1) + Handcrafted({X_hand.shape[1]})'

print(f'Feature set      : {feature_description}')
print(f'Fused shape      : {X_fused.shape}')

In [ ]:
# ── Handcrafted Feature Correlation Heatmap ────────────────────────────────────
hand_df = pd.DataFrame(X_hand, columns=FEATURE_NAMES_HANDCRAFTED)
hand_df['label'] = labels

fig, ax = plt.subplots(figsize=(11, 8))
corr = hand_df.corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, linewidths=0.5, ax=ax, cbar_kws={'shrink': 0.8})
ax.set_title('Handcrafted Feature Correlation Matrix — Decision Classifier',
             fontsize=13, fontweight='bold', pad=15)
plt.tight_layout()
plt.savefig(MODEL_DIR / 'decision_feature_corr.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Feature mean by class ──────────────────────────────────────────────────────
# Shows which handcrafted features discriminate best.
feat_means = hand_df.groupby('label').mean().T
feat_means.columns = ['Not Decision', 'Decision']
feat_means['diff'] = feat_means['Decision'] - feat_means['Not Decision']
feat_means = feat_means.sort_values('diff', ascending=False)

fig, ax = plt.subplots(figsize=(9, 6))
colors = [PALETTE[1] if d > 0 else PALETTE[3] for d in feat_means['diff']]
ax.barh(feat_means.index[::-1], feat_means['diff'][::-1],
        color=colors[::-1], edgecolor='white')
ax.axvline(0, color='black', linewidth=0.8, linestyle='--')
ax.set_xlabel('Mean Difference  (Decision minus Not Decision)', fontsize=11)
ax.set_title('Handcrafted Feature Discriminability\n(positive = higher in Decision class)',
             fontsize=13, fontweight='bold')
legend_elements = [
    Patch(facecolor=PALETTE[1], label='Higher in Decision'),
    Patch(facecolor=PALETTE[3], label='Higher in Not Decision'),
]
ax.legend(handles=legend_elements, fontsize=10)
plt.tight_layout()
plt.savefig(MODEL_DIR / 'decision_feature_discriminability.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Train / Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_fused, labels, test_size=0.20, random_state=SEED, stratify=labels
)

pos_train = y_train.sum()
neg_train = (y_train == 0).sum()
pos_weight = neg_train / pos_train  # used by XGBoost scale_pos_weight

print(f'Train : {X_train.shape[0]} samples  (pos={pos_train}, neg={neg_train})')
print(f'Test  : {X_test.shape[0]}  samples  (pos={y_test.sum()}, neg={(y_test==0).sum()})')
print(f'Positive-weight factor (for XGBoost) : {pos_weight:.2f}')

## 5. Model Training

Eight models trained on the fused feature matrix.
Cross-validation (5-fold, stratified) is used for robust F1 estimation.

| Model | Key advantage |
|---|---|
| Logistic Regression | Fast, interpretable coefficients |
| Random Forest | Robust ensemble baseline |
| Decision Tree | Single tree for explainability |
| KNN | Instance-based; no training phase |
| SVM (RBF) | Strong margin-based classifier |
| XGBoost | Gradient boosting — usually top performer |
| LightGBM | Faster gradient boosting for large feature sets |
| Stacking Ensemble | Meta-learns optimal combination of base models |

In [ ]:
MODELS = {
    'Logistic Regression': LogisticRegression(
        C=1.0, max_iter=1000, class_weight='balanced', random_state=SEED
    ),
    'Random Forest': RandomForestClassifier(
        n_estimators=300, max_depth=None, min_samples_leaf=2,
        class_weight='balanced', random_state=SEED, n_jobs=-1
    ),
    'Decision Tree': DecisionTreeClassifier(
        max_depth=12, min_samples_leaf=3,
        class_weight='balanced', random_state=SEED
    ),
    'KNN': KNeighborsClassifier(
        n_neighbors=7, metric='cosine', n_jobs=-1
    ),
    'SVM': SVC(
        kernel='rbf', C=1.0, probability=True,
        class_weight='balanced', random_state=SEED
    ),
}

if HAS_XGB:
    MODELS['XGBoost'] = XGBClassifier(
        n_estimators=300, max_depth=6, learning_rate=0.05,
        scale_pos_weight=pos_weight, subsample=0.8, colsample_bytree=0.8,
        eval_metric='logloss', random_state=SEED, n_jobs=-1,
    )

if HAS_LGBM:
    MODELS['LightGBM'] = LGBMClassifier(
        n_estimators=300, max_depth=6, learning_rate=0.05,
        class_weight='balanced', subsample=0.8, colsample_bytree=0.8,
        random_state=SEED, n_jobs=-1, verbose=-1,
    )

# ── Stacking Ensemble ──────────────────────────────────────────────────────────
# Base learners feed their probability predictions to a meta-LR which learns
# the optimal combination.  Uses 5-fold out-of-fold predictions so the meta-
# learner never sees training predictions on data it was trained on.
_base = [
    ('lr',  LogisticRegression(C=1.0, max_iter=1000, class_weight='balanced', random_state=SEED)),
    ('rf',  RandomForestClassifier(n_estimators=200, class_weight='balanced', random_state=SEED, n_jobs=-1)),
    ('svm', SVC(kernel='rbf', C=1.0, probability=True, class_weight='balanced', random_state=SEED)),
]
if HAS_XGB:
    _base.append(('xgb', XGBClassifier(
        n_estimators=200, max_depth=5, learning_rate=0.05,
        scale_pos_weight=pos_weight, eval_metric='logloss', random_state=SEED
    )))

MODELS['Stacking Ensemble'] = StackingClassifier(
    estimators=_base,
    final_estimator=LogisticRegression(C=0.5, random_state=SEED),
    cv=5,
    n_jobs=-1,
)

print(f'Models to train ({len(MODELS)}): {list(MODELS.keys())}')

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)


def evaluate_model(name, model, X_tr, y_tr, X_te, y_te):
    model.fit(X_tr, y_tr)
    y_pred = model.predict(X_te)
    y_prob = model.predict_proba(X_te)[:, 1] if hasattr(model, 'predict_proba') else None

    cv_f1 = cross_val_score(model, X_tr, y_tr, cv=cv, scoring='f1', n_jobs=-1)

    pr_auc = average_precision_score(y_te, y_prob) if y_prob is not None else np.nan
    mcc    = matthews_corrcoef(y_te, y_pred)

    return {
        'model'      : model,
        'name'       : name,
        'y_pred'     : y_pred,
        'y_prob'     : y_prob,
        'accuracy'   : accuracy_score(y_te, y_pred),
        'precision'  : precision_score(y_te, y_pred, zero_division=0),
        'recall'     : recall_score(y_te, y_pred, zero_division=0),
        'f1'         : f1_score(y_te, y_pred, zero_division=0),
        'auc'        : roc_auc_score(y_te, y_prob) if y_prob is not None else np.nan,
        'pr_auc'     : pr_auc,
        'mcc'        : mcc,
        'cv_f1_mean' : cv_f1.mean(),
        'cv_f1_std'  : cv_f1.std(),
        'cm'         : confusion_matrix(y_te, y_pred),
    }


results = {}
for name, model in MODELS.items():
    print(f'Training {name}...', end=' ')
    res = evaluate_model(name, model, X_train, y_train, X_test, y_test)
    results[name] = res
    print(f'F1={res["f1"]:.4f}  AUC={res["auc"]:.4f}  PR-AUC={res["pr_auc"]:.4f}  MCC={res["mcc"]:.4f}  CV-F1={res["cv_f1_mean"]:.4f}±{res["cv_f1_std"]:.4f}')

## 6. Results & Visualisations

### 6.1 Metrics Comparison Table

In [ ]:
metrics_df = pd.DataFrame([
    {
        'Model'        : r['name'],
        'Accuracy'     : round(r['accuracy'],   4),
        'Precision'    : round(r['precision'],  4),
        'Recall'       : round(r['recall'],     4),
        'F1'           : round(r['f1'],         4),
        'AUC-ROC'      : round(r['auc'],        4),
        'PR-AUC'       : round(r['pr_auc'],     4),
        'MCC'          : round(r['mcc'],        4),
        'CV F1 (mean)' : round(r['cv_f1_mean'], 4),
        'CV F1 (std)'  : round(r['cv_f1_std'],  4),
    }
    for r in results.values()
]).set_index('Model').sort_values('F1', ascending=False)

display(metrics_df.style
    .background_gradient(cmap='YlGn',   subset=['F1', 'AUC-ROC', 'PR-AUC', 'CV F1 (mean)'])
    .background_gradient(cmap='Purples', subset=['MCC'])
    .background_gradient(cmap='Oranges', subset=['Precision', 'Recall'])
    .format('{:.4f}')
    .set_caption('Decision Classifier — Model Comparison')
)

### 6.2 Confusion Matrices

In [ ]:
n_models = len(results)
n_cols   = 4
n_rows   = (n_models + n_cols - 1) // n_cols
model_colors = PALETTE[:n_models]

fig, axes = plt.subplots(n_rows, n_cols, figsize=(14, 4 * n_rows))
axes = np.array(axes).flatten()

for i, (name, res) in enumerate(results.items()):
    ax  = axes[i]
    cm  = res['cm']
    cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)
    annot = np.array([[f'{cm[r,c]}\n({cm_norm[r,c]:.1%})' for c in range(2)] for r in range(2)])
    sns.heatmap(
        cm_norm, annot=annot, fmt='', cmap='Blues',
        xticklabels=['Not Decision', 'Decision'],
        yticklabels=['Not Decision', 'Decision'],
        ax=ax, cbar=False, linewidths=1, linecolor='white',
        annot_kws={'size': 10}
    )
    ax.set_title(f'{name}\nF1={res["f1"]:.4f}  MCC={res["mcc"]:.4f}',
                 fontsize=10, fontweight='bold', pad=6)
    ax.set_xlabel('Predicted', fontsize=9)
    ax.set_ylabel('Actual', fontsize=9)

for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

fig.suptitle('Confusion Matrices — Decision Classifier',
             fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig(MODEL_DIR / 'decision_confusion_matrices.png', dpi=150, bbox_inches='tight')
plt.show()

### 6.3 ROC Curves

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
ax.plot([0, 1], [0, 1], 'k--', linewidth=1.2, label='Random (AUC = 0.50)', alpha=0.6)

for (name, res), color in zip(results.items(), PALETTE):
    if res['y_prob'] is not None:
        fpr, tpr, _ = roc_curve(y_test, res['y_prob'])
        ax.plot(fpr, tpr, color=color, linewidth=2.2,
                label=f'{name}  (AUC={res["auc"]:.4f})')

ax.set_xlabel('False Positive Rate', fontsize=12)
ax.set_ylabel('True Positive Rate', fontsize=12)
ax.set_title('ROC Curves — Decision Classifier', fontsize=14, fontweight='bold')
ax.legend(loc='lower right', fontsize=9, framealpha=0.9)
ax.set_xlim([0, 1])
ax.set_ylim([0, 1.02])
plt.tight_layout()
plt.savefig(MODEL_DIR / 'decision_roc_curves.png', dpi=150, bbox_inches='tight')
plt.show()

### 6.4 Precision-Recall Curves

PR curves are more informative than ROC curves when classes are imbalanced — they focus entirely on the minority (decision) class.

In [ ]:
baseline_precision = labels.mean()

fig, ax = plt.subplots(figsize=(8, 6))
ax.axhline(baseline_precision, color='k', linestyle='--', linewidth=1.2,
           label=f'Random baseline (P={baseline_precision:.3f})', alpha=0.6)

for (name, res), color in zip(results.items(), PALETTE):
    if res['y_prob'] is not None:
        prec, rec, _ = precision_recall_curve(y_test, res['y_prob'])
        ax.plot(rec, prec, color=color, linewidth=2.2,
                label=f'{name}  (PR-AUC={res["pr_auc"]:.4f})')

ax.set_xlabel('Recall', fontsize=12)
ax.set_ylabel('Precision', fontsize=12)
ax.set_title('Precision-Recall Curves — Decision Classifier',
             fontsize=14, fontweight='bold')
ax.legend(loc='upper right', fontsize=9, framealpha=0.9)
ax.set_xlim([0, 1])
ax.set_ylim([0, 1.02])
plt.tight_layout()
plt.savefig(MODEL_DIR / 'decision_pr_curves.png', dpi=150, bbox_inches='tight')
plt.show()

### 6.5 Metric Bar Chart

In [ ]:
metric_cols  = ['Accuracy', 'Precision', 'Recall', 'F1', 'AUC-ROC', 'PR-AUC']
plot_df      = metrics_df[metric_cols].reset_index()
plot_melted  = plot_df.melt(id_vars='Model', var_name='Metric', value_name='Score')

fig, ax = plt.subplots(figsize=(15, 5))
sns.barplot(
    data=plot_melted, x='Metric', y='Score', hue='Model',
    palette=PALETTE[:len(results)], ax=ax, edgecolor='white', linewidth=0.8
)
for container in ax.containers:
    ax.bar_label(container, fmt='%.3f', fontsize=6.5, padding=2)

ax.set_ylim(0, 1.18)
ax.set_title('Model Comparison by Metric — Decision Classifier',
             fontsize=14, fontweight='bold')
ax.set_xlabel('')
ax.set_ylabel('Score')
ax.legend(title='Model', bbox_to_anchor=(1.01, 1), loc='upper left', fontsize=9)
plt.tight_layout()
plt.savefig(MODEL_DIR / 'decision_metric_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

### 6.6 Cross-Validation F1 vs Test F1

In [ ]:
names    = list(results.keys())
cv_means = [results[n]['cv_f1_mean'] for n in names]
cv_stds  = [results[n]['cv_f1_std']  for n in names]
test_f1  = [results[n]['f1']         for n in names]

x = np.arange(len(names))
w = 0.35

fig, ax = plt.subplots(figsize=(12, 5))
b1 = ax.bar(x - w / 2, cv_means, w, yerr=cv_stds, capsize=5,
            color=PALETTE[0], alpha=0.85, label='CV F1 (5-fold ± std)', edgecolor='white')
b2 = ax.bar(x + w / 2, test_f1, w,
            color=PALETTE[1], alpha=0.85, label='Test F1', edgecolor='white')
ax.bar_label(b1, fmt='%.3f', fontsize=8, padding=3)
ax.bar_label(b2, fmt='%.3f', fontsize=8, padding=3)
ax.set_xticks(x)
ax.set_xticklabels(names, fontsize=9, rotation=15, ha='right')
ax.set_ylim(0, 1.18)
ax.set_ylabel('F1 Score')
ax.set_title('CV F1 vs Test F1 — Decision Classifier', fontsize=14, fontweight='bold')
ax.legend(fontsize=10)
plt.tight_layout()
plt.savefig(MODEL_DIR / 'decision_cv_vs_test_f1.png', dpi=150, bbox_inches='tight')
plt.show()

### 6.7 MCC Bar Chart

MCC (Matthews Correlation Coefficient) ranges from −1 to +1 and considers all four cells of the confusion matrix, making it the most balanced single metric for imbalanced binary classification.

In [ ]:
mcc_vals = [(n, results[n]['mcc']) for n in names]
mcc_vals.sort(key=lambda x: x[1], reverse=True)
mcc_names, mcc_scores = zip(*mcc_vals)

fig, ax = plt.subplots(figsize=(9, 4))
colors = [PALETTE[0] if v >= 0 else PALETTE[3] for v in mcc_scores]
bars = ax.barh(mcc_names[::-1], mcc_scores[::-1], color=colors[::-1], edgecolor='white')
ax.bar_label(bars, fmt='%.4f', fontsize=9, padding=4)
ax.axvline(0, color='black', linewidth=0.8, linestyle='--')
ax.set_xlabel('MCC Score', fontsize=11)
ax.set_title('Matthews Correlation Coefficient — Decision Classifier',
             fontsize=13, fontweight='bold')
ax.set_xlim(-0.1, 1.0)
plt.tight_layout()
plt.savefig(MODEL_DIR / 'decision_mcc.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Threshold Tuning

Most classifiers default to a 0.5 probability threshold. For imbalanced classes this is rarely optimal. We sweep thresholds to find the one that maximises F1 on the test set for the best model.

In [ ]:
best_name = metrics_df['F1'].idxmax()
best_prob = results[best_name]['y_prob']

if best_prob is not None:
    thresholds = np.arange(0.05, 0.95, 0.01)
    f1_scores  = [f1_score(y_test, (best_prob >= t).astype(int), zero_division=0)
                  for t in thresholds]
    prec_scores = [precision_score(y_test, (best_prob >= t).astype(int), zero_division=0)
                   for t in thresholds]
    rec_scores  = [recall_score(y_test, (best_prob >= t).astype(int), zero_division=0)
                   for t in thresholds]

    opt_idx    = int(np.argmax(f1_scores))
    opt_thresh = thresholds[opt_idx]
    opt_f1     = f1_scores[opt_idx]

    fig, ax = plt.subplots(figsize=(10, 4))
    ax.plot(thresholds, f1_scores,   color=PALETTE[0], linewidth=2.2, label='F1')
    ax.plot(thresholds, prec_scores, color=PALETTE[1], linewidth=1.8, label='Precision', linestyle='--')
    ax.plot(thresholds, rec_scores,  color=PALETTE[2], linewidth=1.8, label='Recall',    linestyle=':')
    ax.axvline(opt_thresh, color=PALETTE[3], linewidth=1.5, linestyle='--',
               label=f'Optimal threshold = {opt_thresh:.2f}  (F1={opt_f1:.4f})')
    ax.axvline(0.5, color='grey', linewidth=1.0, linestyle=':', alpha=0.7, label='Default threshold = 0.50')
    ax.set_xlabel('Probability Threshold', fontsize=11)
    ax.set_ylabel('Score', fontsize=11)
    ax.set_title(f'Threshold Tuning — {best_name}', fontsize=13, fontweight='bold')
    ax.legend(fontsize=9)
    ax.set_xlim([0.05, 0.95])
    ax.set_ylim([0, 1.05])
    plt.tight_layout()
    plt.savefig(MODEL_DIR / 'decision_threshold_tuning.png', dpi=150, bbox_inches='tight')
    plt.show()

    y_pred_tuned = (best_prob >= opt_thresh).astype(int)
    default_f1   = results[best_name]['f1']
    print(f'Best model       : {best_name}')
    print(f'Default F1 (0.50): {default_f1:.4f}')
    print(f'Tuned   F1 ({opt_thresh:.2f}) : {opt_f1:.4f}  (improvement: +{opt_f1 - default_f1:.4f})')
else:
    print(f'{best_name} does not support predict_proba — threshold tuning skipped.')

## 8. Feature Group Ablation

Train the best model on progressively richer feature sets to quantify each group's contribution.

In [ ]:
import copy

# Re-build feature variants ────────────────────────────────────────────────────
feat_sets = {}

# Set A — TF-IDF only (simplest baseline)
feat_sets['TF-IDF only'] = X_tfidf

# Set B — TF-IDF + Handcrafted (same as Importance Classifier)
feat_sets['TF-IDF + Handcrafted'] = hstack([X_tfidf, hand_col])

# Set C — TF-IDF + Handcrafted + BM25
feat_sets['TF-IDF + Handcrafted + BM25'] = hstack([X_tfidf, hand_col, bm25_col])

# Set D — full (includes SBERT if available)
feat_sets['Full Feature Set'] = X_fused

# Use best model (re-instantiated to avoid data leakage)
ablation_model_cls = copy.deepcopy(MODELS[best_name])
ablation_results = {}

for feat_name, X_feat in feat_sets.items():
    Xtr, Xte, ytr, yte = train_test_split(
        X_feat, labels, test_size=0.20, random_state=SEED, stratify=labels
    )
    m = copy.deepcopy(ablation_model_cls)
    m.fit(Xtr, ytr)
    yp = m.predict(Xte)
    yb = m.predict_proba(Xte)[:, 1] if hasattr(m, 'predict_proba') else None
    ablation_results[feat_name] = {
        'f1'     : f1_score(yte, yp, zero_division=0),
        'pr_auc' : average_precision_score(yte, yb) if yb is not None else np.nan,
        'mcc'    : matthews_corrcoef(yte, yp),
        'n_feat' : X_feat.shape[1],
    }
    print(f'{feat_name:<40}  F1={ablation_results[feat_name]["f1"]:.4f}  PR-AUC={ablation_results[feat_name]["pr_auc"]:.4f}  #feat={X_feat.shape[1]}')

In [ ]:
abl_names  = list(ablation_results.keys())
abl_f1     = [ablation_results[k]['f1']     for k in abl_names]
abl_prauc  = [ablation_results[k]['pr_auc'] for k in abl_names]
abl_mcc    = [ablation_results[k]['mcc']    for k in abl_names]

x = np.arange(len(abl_names))
w = 0.25

fig, ax = plt.subplots(figsize=(12, 5))
b1 = ax.bar(x - w,     abl_f1,    w, color=PALETTE[0], alpha=0.88, label='F1',     edgecolor='white')
b2 = ax.bar(x,         abl_prauc, w, color=PALETTE[1], alpha=0.88, label='PR-AUC', edgecolor='white')
b3 = ax.bar(x + w,     abl_mcc,   w, color=PALETTE[2], alpha=0.88, label='MCC',    edgecolor='white')
ax.bar_label(b1, fmt='%.3f', fontsize=8, padding=2)
ax.bar_label(b2, fmt='%.3f', fontsize=8, padding=2)
ax.bar_label(b3, fmt='%.3f', fontsize=8, padding=2)
ax.set_xticks(x)
ax.set_xticklabels(abl_names, fontsize=9, rotation=10, ha='right')
ax.set_ylim(0, 1.15)
ax.set_ylabel('Score')
ax.set_title(f'Feature Group Ablation — {best_name}', fontsize=13, fontweight='bold')
ax.legend(fontsize=10)
plt.tight_layout()
plt.savefig(MODEL_DIR / 'decision_ablation.png', dpi=150, bbox_inches='tight')
plt.show()

## 9. Feature Importance

### 9.1 Random Forest — Top 25 Features

In [ ]:
# Build the full feature name list matching the fused matrix column order
tfidf_feat_names = tfidf.get_feature_names_out().tolist()
all_feat_names = tfidf_feat_names + ['bm25_score'] + FEATURE_NAMES_HANDCRAFTED
if HAS_SBERT and SBERT_DIM > 0:
    all_feat_names += [f'sbert_{i}' for i in range(SBERT_DIM)]

if 'Random Forest' in results:
    rf_model = results['Random Forest']['model']
    importances = rf_model.feature_importances_
    feat_imp_df = pd.DataFrame({'feature': all_feat_names, 'importance': importances})
    feat_imp_df = feat_imp_df.sort_values('importance', ascending=False).head(25)

    def feat_color(name):
        if name in FEATURE_NAMES_HANDCRAFTED:   return PALETTE[2]
        if name == 'bm25_score':                 return PALETTE[1]
        if name.startswith('sbert_'):            return PALETTE[4]
        return PALETTE[0]

    colors = [feat_color(f) for f in feat_imp_df['feature']]

    fig, ax = plt.subplots(figsize=(9, 8))
    ax.barh(feat_imp_df['feature'][::-1], feat_imp_df['importance'][::-1],
            color=colors[::-1], edgecolor='white')
    ax.set_xlabel('Feature Importance (Mean Decrease Impurity)', fontsize=11)
    ax.set_title('Random Forest — Top 25 Feature Importances',
                 fontsize=13, fontweight='bold')
    legend_elements = [
        Patch(facecolor=PALETTE[0], label='TF-IDF token'),
        Patch(facecolor=PALETTE[1], label='BM25 score'),
        Patch(facecolor=PALETTE[2], label='Handcrafted feature'),
        Patch(facecolor=PALETTE[4], label='SBERT dimension'),
    ]
    ax.legend(handles=legend_elements, loc='lower right', fontsize=10)
    plt.tight_layout()
    plt.savefig(MODEL_DIR / 'decision_rf_feature_importance.png', dpi=150, bbox_inches='tight')
    plt.show()

### 9.2 Logistic Regression — Top Positive & Negative Coefficients

In [ ]:
if 'Logistic Regression' in results:
    lr_model = results['Logistic Regression']['model']
    coefs    = lr_model.coef_[0]

    coef_df  = pd.DataFrame({'feature': all_feat_names, 'coef': coefs})
    top_pos  = coef_df.nlargest(15, 'coef')
    top_neg  = coef_df.nsmallest(15, 'coef')
    top_lr   = pd.concat([top_pos, top_neg]).sort_values('coef')

    fig, ax = plt.subplots(figsize=(9, 9))
    colors  = [PALETTE[1] if c > 0 else PALETTE[3] for c in top_lr['coef']]
    ax.barh(top_lr['feature'], top_lr['coef'], color=colors, edgecolor='white')
    ax.axvline(0, color='black', linewidth=0.8, linestyle='--')
    ax.set_xlabel('Coefficient Value', fontsize=11)
    ax.set_title('Logistic Regression — Top 15 Positive & Negative Coefficients\n'
                 '(orange → Decision, red → Not Decision)',
                 fontsize=13, fontweight='bold')
    legend_elements = [
        Patch(facecolor=PALETTE[1], label='Predicts Decision'),
        Patch(facecolor=PALETTE[3], label='Predicts Not Decision'),
    ]
    ax.legend(handles=legend_elements, loc='lower right', fontsize=10)
    plt.tight_layout()
    plt.savefig(MODEL_DIR / 'decision_lr_coefficients.png', dpi=150, bbox_inches='tight')
    plt.show()

### 9.3 LightGBM — Feature Importances

In [ ]:
if HAS_LGBM and 'LightGBM' in results:
    lgbm_model = results['LightGBM']['model']
    lgbm_imp   = lgbm_model.feature_importances_

    lgbm_df = pd.DataFrame({'feature': all_feat_names, 'importance': lgbm_imp})
    lgbm_df = lgbm_df.sort_values('importance', ascending=False).head(25)

    def feat_color(name):
        if name in FEATURE_NAMES_HANDCRAFTED: return PALETTE[2]
        if name == 'bm25_score':              return PALETTE[1]
        if name.startswith('sbert_'):         return PALETTE[4]
        return PALETTE[0]

    colors = [feat_color(f) for f in lgbm_df['feature']]

    fig, ax = plt.subplots(figsize=(9, 8))
    ax.barh(lgbm_df['feature'][::-1], lgbm_df['importance'][::-1],
            color=colors[::-1], edgecolor='white')
    ax.set_xlabel('Feature Importance (Split count)', fontsize=11)
    ax.set_title('LightGBM — Top 25 Feature Importances', fontsize=13, fontweight='bold')
    legend_elements = [
        Patch(facecolor=PALETTE[0], label='TF-IDF token'),
        Patch(facecolor=PALETTE[1], label='BM25 score'),
        Patch(facecolor=PALETTE[2], label='Handcrafted feature'),
        Patch(facecolor=PALETTE[4], label='SBERT dimension'),
    ]
    ax.legend(handles=legend_elements, loc='lower right', fontsize=10)
    plt.tight_layout()
    plt.savefig(MODEL_DIR / 'decision_lgbm_feature_importance.png', dpi=150, bbox_inches='tight')
    plt.show()

## 10. Classification Reports

In [ ]:
for name, res in results.items():
    sep = '=' * 60
    print(sep)
    print(f'  {name}')
    print(sep)
    print(classification_report(
        y_test, res['y_pred'],
        target_names=['Not Decision', 'Decision']
    ))

## 11. Best Model Selection & Save

In [ ]:
best_name = metrics_df['F1'].idxmax()
best_res  = results[best_name]

print('=' * 55)
print(f'  Best Model : {best_name}')
print('=' * 55)
print(f'  F1         : {best_res["f1"]:.4f}')
print(f'  AUC-ROC    : {best_res["auc"]:.4f}')
print(f'  PR-AUC     : {best_res["pr_auc"]:.4f}')
print(f'  MCC        : {best_res["mcc"]:.4f}')
print(f'  Precision  : {best_res["precision"]:.4f}')
print(f'  Recall     : {best_res["recall"]:.4f}')
print(f'  Accuracy   : {best_res["accuracy"]:.4f}')
print('=' * 55)

save_obj = {
    'model'                : best_res['model'],
    'tfidf_vectorizer'     : tfidf,
    'feature_names'        : all_feat_names,
    'handcrafted_fn_names' : FEATURE_NAMES_HANDCRAFTED,
    'has_bm25'             : HAS_BM25,
    'has_sbert'            : HAS_SBERT,
    'sbert_dim'            : SBERT_DIM,
    'optimal_threshold'    : opt_thresh if 'opt_thresh' in dir() else 0.5,
    'best_model_name'      : best_name,
    'metrics': {
        'f1'       : best_res['f1'],
        'auc'      : best_res['auc'],
        'pr_auc'   : best_res['pr_auc'],
        'mcc'      : best_res['mcc'],
        'precision': best_res['precision'],
        'recall'   : best_res['recall'],
        'accuracy' : best_res['accuracy'],
    },
}

save_path = MODEL_DIR / 'decision_classifier.pkl'
with open(save_path, 'wb') as f:
    pickle.dump(save_obj, f)

print(f'\nModel saved → {save_path}')

## 13. BERT Fine-Tuning

Fine-tuning updates **all 110 M parameters** of `bert-base-uncased` plus a 2-class head
on your exact decision dataset — giving it the highest accuracy ceiling of any approach here.

| Step | Detail |
|---|---|
| Model | `bert-base-uncased` (12 layers, 768 hidden, 12 heads) |
| Max tokens | 128 (covers virtually all AMI sentences) |
| Epochs | 5 with early stopping (patience = 2 on val F1) |
| Batch size | 16 train / 32 eval |
| Learning rate | 2e-5 with linear warm-up (10 %) |
| Imbalance | Weighted cross-entropy loss — minority class gets `neg/pos` weight |
| Requires | `pip install transformers torch` · GPU recommended, CPU works |

In [ ]:
# ── BERT availability check ───────────────────────────────────────────────────
try:
    import torch
    from transformers import (
        AutoTokenizer, AutoModelForSequenceClassification,
        TrainingArguments, Trainer,
    )
    from torch.utils.data import Dataset as TorchDataset
    HAS_TRANSFORMERS = True
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    print(f'transformers available  |  device: {device}')
    print(f'torch version : {torch.__version__}')
    import transformers as _tr
    print(f'transformers  : {_tr.__version__}')
except ImportError as e:
    HAS_TRANSFORMERS = False
    print(f'Missing package: {e}')
    print('Run:  pip install transformers torch')
    print('BERT fine-tuning section will be skipped.')

In [ ]:
if HAS_TRANSFORMERS:
    # ── Reproduce the same 80/20 stratified split (text version) ─────────────
    all_idx = np.arange(len(texts))
    tr_idx, te_idx = train_test_split(
        all_idx, test_size=0.20, random_state=SEED, stratify=labels
    )
    texts_bert_tr = [texts[i] for i in tr_idx]
    texts_bert_te = [texts[i] for i in te_idx]
    y_bert_tr     = labels[tr_idx]
    y_bert_te     = labels[te_idx]

    BERT_MODEL_NAME = 'bert-base-uncased'
    bert_tokenizer  = AutoTokenizer.from_pretrained(BERT_MODEL_NAME)

    class DecisionDataset(TorchDataset):
        """Tokenises sentences and wraps them as a PyTorch Dataset."""
        def __init__(self, texts, labels, tokenizer, max_len=128):
            self.enc    = tokenizer(
                texts, truncation=True, padding='max_length',
                max_length=max_len, return_tensors='pt',
            )
            self.labels = torch.tensor(labels, dtype=torch.long)

        def __len__(self):
            return len(self.labels)

        def __getitem__(self, idx):
            item = {k: v[idx] for k, v in self.enc.items()}
            item['labels'] = self.labels[idx]
            return item

    bert_train_ds = DecisionDataset(texts_bert_tr, y_bert_tr, bert_tokenizer)
    bert_test_ds  = DecisionDataset(texts_bert_te, y_bert_te, bert_tokenizer)

    print(f'Train : {len(bert_train_ds)} samples  '
          f'(pos={y_bert_tr.sum()}, neg={(y_bert_tr==0).sum()})')
    print(f'Test  : {len(bert_test_ds)}  samples  '
          f'(pos={y_bert_te.sum()}, neg={(y_bert_te==0).sum()})')
    print(f'Max sequence length: 128 tokens')

In [ ]:
if HAS_TRANSFORMERS:
    # ── Weighted cross-entropy to handle class imbalance ──────────────────────
    _bert_pw = float((y_bert_tr == 0).sum() / y_bert_tr.sum())

    def compute_bert_metrics(eval_pred):
        logits, lbls = eval_pred
        preds   = np.argmax(logits, axis=1)
        shifted = logits - logits.max(axis=1, keepdims=True)  # numerical stability
        probs   = np.exp(shifted[:, 1]) / np.exp(shifted).sum(axis=1)
        return {
            'f1'    : float(f1_score(lbls, preds, zero_division=0)),
            'pr_auc': float(average_precision_score(lbls, probs)),
        }

    class WeightedTrainer(Trainer):
        """Trainer subclass that applies class-weighted cross-entropy loss."""
        def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
            labels  = inputs.pop('labels')
            outputs = model(**inputs)
            weights = torch.tensor([1.0, _bert_pw],
                                   device=outputs.logits.device, dtype=torch.float)
            loss = torch.nn.CrossEntropyLoss(weight=weights)(outputs.logits, labels)
            return (loss, outputs) if return_outputs else loss

    bert_model = AutoModelForSequenceClassification.from_pretrained(
        BERT_MODEL_NAME, num_labels=2
    )

    BERT_OUTPUT = str(MODEL_DIR / 'bert_decision')

    # evaluation_strategy was renamed to eval_strategy in transformers >= 4.41
    # Try both for compatibility.
    import inspect
    _ta_params = inspect.signature(TrainingArguments.__init__).parameters
    _eval_kw   = 'eval_strategy' if 'eval_strategy' in _ta_params else 'evaluation_strategy'

    training_args = TrainingArguments(
        output_dir                  = BERT_OUTPUT,
        num_train_epochs            = 5,
        per_device_train_batch_size = 16,
        per_device_eval_batch_size  = 32,
        learning_rate               = 2e-5,
        weight_decay                = 0.01,
        warmup_ratio                = 0.1,
        save_strategy               = 'epoch',
        load_best_model_at_end      = True,
        metric_for_best_model       = 'f1',
        greater_is_better           = True,
        logging_steps               = 20,
        report_to                   = 'none',
        seed                        = SEED,
        **{_eval_kw: 'epoch'},
    )

    bert_trainer = WeightedTrainer(
        model           = bert_model,
        args            = training_args,
        train_dataset   = bert_train_ds,
        eval_dataset    = bert_test_ds,
        compute_metrics = compute_bert_metrics,
    )

    print('Starting BERT fine-tuning...')
    print(f'  Model      : {BERT_MODEL_NAME}')
    print(f'  Epochs     : {training_args.num_train_epochs}')
    print(f'  Batch size : {training_args.per_device_train_batch_size}')
    print(f'  LR         : {training_args.learning_rate}')
    print(f'  Pos weight : {_bert_pw:.2f}')
    print(f'  Device     : {device}')
    bert_trainer.train()
    print('\nFine-tuning complete.')

In [ ]:
if HAS_TRANSFORMERS:
    # ── Evaluate BERT on the test set ─────────────────────────────────────────
    bert_out    = bert_trainer.predict(bert_test_ds)
    bert_logits = bert_out.predictions
    bert_lbls   = bert_out.label_ids

    bert_preds  = np.argmax(bert_logits, axis=1)
    shifted     = bert_logits - bert_logits.max(axis=1, keepdims=True)
    bert_probs  = np.exp(shifted[:, 1]) / np.exp(shifted).sum(axis=1)

    bert_f1     = f1_score(bert_lbls,  bert_preds, zero_division=0)
    bert_prec   = precision_score(bert_lbls, bert_preds, zero_division=0)
    bert_rec    = recall_score(bert_lbls, bert_preds, zero_division=0)
    bert_acc    = accuracy_score(bert_lbls, bert_preds)
    bert_auc    = roc_auc_score(bert_lbls, bert_probs)
    bert_pr_auc = average_precision_score(bert_lbls, bert_probs)
    bert_mcc    = matthews_corrcoef(bert_lbls, bert_preds)
    bert_cm     = confusion_matrix(bert_lbls, bert_preds)

    # Store in the global results dict so the comparison section picks it up
    results['BERT (fine-tuned)'] = {
        'model'      : bert_model,
        'name'       : 'BERT (fine-tuned)',
        'y_pred'     : bert_preds,
        'y_prob'     : bert_probs,
        'accuracy'   : bert_acc,
        'precision'  : bert_prec,
        'recall'     : bert_rec,
        'f1'         : bert_f1,
        'auc'        : bert_auc,
        'pr_auc'     : bert_pr_auc,
        'mcc'        : bert_mcc,
        'cv_f1_mean' : bert_f1,
        'cv_f1_std'  : 0.0,
        'cm'         : bert_cm,
    }

    print('BERT fine-tuned — test metrics')
    print(f'  F1         : {bert_f1:.4f}')
    print(f'  AUC-ROC    : {bert_auc:.4f}')
    print(f'  PR-AUC     : {bert_pr_auc:.4f}')
    print(f'  MCC        : {bert_mcc:.4f}')
    print(f'  Precision  : {bert_prec:.4f}')
    print(f'  Recall     : {bert_rec:.4f}')
    print(f'  Accuracy   : {bert_acc:.4f}')

    # ── Save fine-tuned BERT ───────────────────────────────────────────────────
    bert_trainer.save_model(BERT_OUTPUT)
    bert_tokenizer.save_pretrained(BERT_OUTPUT)
    print(f'\nBERT model saved → {BERT_OUTPUT}')
else:
    print('BERT fine-tuning was skipped (transformers / torch not installed).')
    print('The comparison section below will show only the classical ML models.')

## 14. Final Model Comparison

Head-to-head comparison of **every** model trained in this notebook:
classical ML baselines, the enhanced feature-engineered models, and BERT fine-tuning.
BERT rows are highlighted in the table and marked with a star in charts.

In [ ]:
# ── Build the final unified metrics table ─────────────────────────────────
final_df = pd.DataFrame([
    {
        'Model'     : r['name'],
        'Accuracy'  : round(r['accuracy'],   4),
        'Precision' : round(r['precision'],  4),
        'Recall'    : round(r['recall'],     4),
        'F1'        : round(r['f1'],         4),
        'AUC-ROC'   : round(r['auc'],        4),
        'PR-AUC'    : round(r['pr_auc'],     4),
        'MCC'       : round(r['mcc'],        4),
    }
    for r in results.values()
]).set_index('Model').sort_values('F1', ascending=False)

def _highlight_bert(row):
    """Bold gold background for BERT row."""
    if 'BERT' in row.name:
        return ['background-color: #fff3cd; font-weight: bold'] * len(row)
    return [''] * len(row)

display(final_df.style
    .background_gradient(cmap='YlGn',    subset=['F1', 'AUC-ROC', 'PR-AUC'])
    .background_gradient(cmap='Purples', subset=['MCC'])
    .background_gradient(cmap='Oranges', subset=['Precision', 'Recall'])
    .apply(_highlight_bert, axis=1)
    .format('{:.4f}')
    .set_caption('All Models — Final Comparison  (sorted by F1, BERT highlighted in gold)')
)

In [ ]:
# ── Grouped bar chart — F1, PR-AUC, MCC for all models ───────────────────
model_names_all = list(final_df.index)
is_bert         = ['BERT' in n for n in model_names_all]

f1_vals     = final_df['F1'].tolist()
pr_auc_vals = final_df['PR-AUC'].tolist()
mcc_vals    = final_df['MCC'].tolist()

x = np.arange(len(model_names_all))
w = 0.26

fig, ax = plt.subplots(figsize=(max(12, len(model_names_all) * 1.5), 6))

b1 = ax.bar(x - w, f1_vals,     w, color=PALETTE[0], alpha=0.88, label='F1',     edgecolor='white')
b2 = ax.bar(x,     pr_auc_vals, w, color=PALETTE[1], alpha=0.88, label='PR-AUC', edgecolor='white')
b3 = ax.bar(x + w, mcc_vals,    w, color=PALETTE[2], alpha=0.88, label='MCC',    edgecolor='white')

ax.bar_label(b1, fmt='%.3f', fontsize=7, padding=2)
ax.bar_label(b2, fmt='%.3f', fontsize=7, padding=2)
ax.bar_label(b3, fmt='%.3f', fontsize=7, padding=2)

# Mark BERT bars with a gold outline
for bars in [b1, b2, b3]:
    for bar, is_b in zip(bars, is_bert):
        if is_b:
            bar.set_edgecolor('goldenrod')
            bar.set_linewidth(2.5)

ax.set_xticks(x)
ax.set_xticklabels(model_names_all, fontsize=9, rotation=20, ha='right')
ax.set_ylim(0, 1.18)
ax.set_ylabel('Score')
ax.set_title('Final Model Comparison — F1 / PR-AUC / MCC\n'
             '(gold outline = BERT fine-tuned)',
             fontsize=14, fontweight='bold')
ax.legend(fontsize=10, loc='upper right')
ax.axhline(0, color='black', linewidth=0.5)
plt.tight_layout()
plt.savefig(MODEL_DIR / 'decision_final_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Combined ROC curves including BERT ────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# ROC
axes[0].plot([0, 1], [0, 1], 'k--', linewidth=1.2, label='Random (AUC=0.50)', alpha=0.6)
for (name, res), color in zip(results.items(), PALETTE * 3):
    if res['y_prob'] is not None:
        fpr, tpr, _ = roc_curve(y_test if 'BERT' not in name else bert_lbls
                                 if 'bert_lbls' in dir() else y_test,
                                res['y_prob'])
        lw  = 3.0 if 'BERT' in name else 1.8
        ls  = '-'
        axes[0].plot(fpr, tpr, color=color, linewidth=lw, linestyle=ls,
                    label=f'{name}  (AUC={res["auc"]:.4f})')
axes[0].set_title('ROC Curves — All Models', fontsize=13, fontweight='bold')
axes[0].set_xlabel('False Positive Rate')
axes[0].set_ylabel('True Positive Rate')
axes[0].legend(fontsize=7, loc='lower right', framealpha=0.9)
axes[0].set_xlim([0, 1]); axes[0].set_ylim([0, 1.02])

# PR
baseline_p = labels.mean()
axes[1].axhline(baseline_p, color='k', linestyle='--', linewidth=1.2,
               label=f'Random baseline (P={baseline_p:.3f})', alpha=0.6)
for (name, res), color in zip(results.items(), PALETTE * 3):
    if res['y_prob'] is not None:
        y_true_here = bert_lbls if ('BERT' in name and 'bert_lbls' in dir()) else y_test
        prec, rec, _ = precision_recall_curve(y_true_here, res['y_prob'])
        lw = 3.0 if 'BERT' in name else 1.8
        axes[1].plot(rec, prec, color=color, linewidth=lw,
                    label=f'{name}  (PR-AUC={res["pr_auc"]:.4f})')
axes[1].set_title('PR Curves — All Models', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Recall')
axes[1].set_ylabel('Precision')
axes[1].legend(fontsize=7, loc='upper right', framealpha=0.9)
axes[1].set_xlim([0, 1]); axes[1].set_ylim([0, 1.02])

plt.tight_layout()
plt.savefig(MODEL_DIR / 'decision_final_roc_pr.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Print the final winner ─────────────────────────────────────────────────
winner_name = final_df['F1'].idxmax()
winner      = results[winner_name]

rank_df = final_df[['F1', 'PR-AUC', 'MCC']].copy()
rank_df['Rank (F1)']     = rank_df['F1'].rank(ascending=False).astype(int)
rank_df['Rank (PR-AUC)'] = rank_df['PR-AUC'].rank(ascending=False).astype(int)
rank_df['Rank (MCC)']    = rank_df['MCC'].rank(ascending=False).astype(int)

print('=' * 65)
print(f'  BEST MODEL: {winner_name}')
print('=' * 65)
print(f'  F1         : {winner["f1"]:.4f}')
print(f'  AUC-ROC    : {winner["auc"]:.4f}')
print(f'  PR-AUC     : {winner["pr_auc"]:.4f}')
print(f'  MCC        : {winner["mcc"]:.4f}')
print(f'  Precision  : {winner["precision"]:.4f}')
print(f'  Recall     : {winner["recall"]:.4f}')
print(f'  Accuracy   : {winner["accuracy"]:.4f}')
print('=' * 65)
print()
print('Ranking summary (1 = best):')
display(rank_df.sort_values('Rank (F1)'))

## 15. Inference Helper

Quick sanity check — feed new sentences through the saved model.

In [ ]:
def predict_decision(sentences: list, model_path=save_path,
                     use_optimal_threshold: bool = True) -> pd.DataFrame:
    """Load saved model and predict decision label for a list of sentences."""
    with open(model_path, 'rb') as f:
        obj = pickle.load(f)

    clf    = obj['model']
    tvec   = obj['tfidf_vectorizer']
    thresh = obj['optimal_threshold'] if use_optimal_threshold else 0.5

    X_tf   = tvec.transform(sentences)
    X_hc   = extract_handcrafted(sentences)
    X_hc   = StandardScaler().fit_transform(X_hc)

    if HAS_BM25:
        tokenized = [s.lower().split() for s in sentences]
        bm25_idx  = BM25Okapi(tokenized)
        bm25_sc   = bm25_idx.get_scores(DECISION_QUERY)
        bm25_sc   = (bm25_sc - bm25_sc.min()) / (bm25_sc.max() - bm25_sc.min() + 1e-9)
        bm25_feat = csr_matrix(bm25_sc.reshape(-1, 1))
    else:
        bm25_feat = csr_matrix(np.zeros((len(sentences), 1)))

    if HAS_SBERT and obj['has_sbert'] and obj['sbert_dim'] > 0:
        sbert_m = SentenceTransformer('all-MiniLM-L6-v2')
        X_sb    = sbert_m.encode(sentences, convert_to_numpy=True)
        X       = hstack([X_tf, bm25_feat, csr_matrix(X_hc), csr_matrix(X_sb)])
    else:
        X       = hstack([X_tf, bm25_feat, csr_matrix(X_hc)])

    probs = clf.predict_proba(X)[:, 1] if hasattr(clf, 'predict_proba') else None
    preds = (probs >= thresh).astype(int) if probs is not None else clf.predict(X)

    return pd.DataFrame({
        'sentence'   : sentences,
        'prediction' : ['Decision' if p == 1 else 'Not Decision' for p in preds],
        'confidence' : [f'{p:.2%}' if probs is not None else '—' for p in (probs if probs is not None else [])],
        'threshold'  : thresh,
    })


test_sentences = [
    "We decided to go with the touchscreen interface.",
    "So we agreed on a budget of fifty euros for the prototype.",
    "The team has confirmed that the remote will use voice control.",
    "Someone mentioned that it was cold outside.",
    "I think we might consider a curved design, maybe.",
    "It was resolved that John would handle the marketing materials.",
    "The meeting started at nine in the morning.",
    "So the final colour is going to be black.",
]

display(predict_decision(test_sentences))